# First-class functions

In Python, functions are values. You can assign a function to a variable, store it in a list, pass it to another function, and return it from another function.

In [60]:
def square(x):
    return x * x

operation = square
print(operation(5))  # 25

def apply_to_list(func, items):
    return [func(item) for item in items]

scores = [0.8, 0.3, 0.95, 0.6]
normalized = apply_to_list(lambda x: round(x * 100), scores)
print(normalized)  # [80, 30, 95, 60]

25
[80, 30, 95, 60]


In [61]:
def by_accuracy(result):
    return result["accuracy"]

def by_latency(result):
    return result["latency_ms"]

results = [
    {"model": "large-reasoning-model", "accuracy": 0.92, "latency_ms": 800},
    {"model": "balanced-model", "accuracy": 0.95, "latency_ms": 1200},
    {"model": "small-local-model", "accuracy": 0.88, "latency_ms": 200},
]

fastest = sorted(results, key=by_latency)
most_accurate = sorted(results, key=by_accuracy, reverse=True)

print(fastest[0]["model"])        # small-local-model
print(most_accurate[0]["model"])  # balanced-model

small-local-model
balanced-model


# *args and **kwargs
*args collects positional arguments into a tuple. **kwargs collects keyword arguments into a dictionary.

In [62]:
def log_and_call(func, *args, **kwargs):
    print(f"Calling {func.__name__} with args={args}, kwargs={kwargs}")
    return func(*args, **kwargs)

def embed_text(text, model="text-embedding-3-small", dimensions=256):
    return f"Embedding '{text}' with {model} ({dimensions}d)"

result = log_and_call(embed_text, "Hello world", model="text-embedding-3-large")
print(result)

Calling embed_text with args=('Hello world',), kwargs={'model': 'text-embedding-3-large'}
Embedding 'Hello world' with text-embedding-3-large (256d)


In [63]:
def call_llm(prompt, **kwargs):
    """Set project defaults while allowing each call to override them."""
    defaults = {
        "model": "fast-default-model",
        "temperature": 0.2,
        "max_output_tokens": 1000,
    }
    config = {**defaults, **kwargs}
    print(f"Calling {config['model']} with temp={config['temperature']}")
    return config

call_llm("Explain RAG")
call_llm("Explain RAG", temperature=0.0, max_output_tokens=500)


Calling fast-default-model with temp=0.2
Calling fast-default-model with temp=0.0


{'model': 'fast-default-model', 'temperature': 0.0, 'max_output_tokens': 500}

# Closures

A closure is a function that remembers values from the scope where it was created, even after that outer function has returned.

In [64]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15

10
15


In [65]:
def make_classifier(threshold):
    def classify(score):
        return "positive" if score >= threshold else "negative"
    return classify

strict = make_classifier(0.9)
lenient = make_classifier(0.5)

scores = [0.45, 0.72, 0.91, 0.88]
print([strict(s) for s in scores])   # ['negative', 'negative', 'positive', 'negative']
print([lenient(s) for s in scores])  # ['negative', 'positive', 'positive', 'positive']

['negative', 'negative', 'positive', 'negative']
['negative', 'positive', 'positive', 'positive']


# Decorators
Decorators are where closures become practical infrastructure.
A decorator is a function that takes another function, wraps it with extra behavior, and returns the wrapped version
The @decorator syntax is shorthand for passing a function through another function.

In [66]:
import time
import time
from functools import wraps

def timing(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.3f}s")
        return result
    return wrapper

def slow_embedding(text):
    time.sleep(0.5)
    return [0.1, 0.2, 0.3]

slow_embedding = timing(slow_embedding)
slow_embedding("hello")  # slow_embedding took 0.501s

@timing
def slow_embedding2(text):
    time.sleep(0.5)
    return [0.1, 0.2, 0.3]

slow_embedding2("hello")  # slow_embedding took 0.500s

slow_embedding took 0.501s
slow_embedding2 took 0.500s


[0.1, 0.2, 0.3]

## The @retry Decorator

In [67]:
import time
from functools import wraps

def retry(max_retries=3, delay=1.0, backoff=2.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            current_delay = delay

            for attempt in range(max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exception = e
                    if attempt < max_retries:
                        print(f"Attempt {attempt + 1} failed: {e}. "
                              f"Retrying in {current_delay:.1f}s...")
                        time.sleep(current_delay)
                        current_delay *= backoff

            raise last_exception
        return wrapper
    return decorator

failures_left = 2

@retry(max_retries=3, delay=0.1, backoff=2.0)
def call_embedding_api(text):
    global failures_left

    if failures_left > 0:
        failures_left -= 1
        raise ConnectionError("API timeout")

    return [0.1, 0.2, 0.3]

result = call_embedding_api("Hello world")
print(result)

Attempt 1 failed: API timeout. Retrying in 0.1s...
Attempt 2 failed: API timeout. Retrying in 0.2s...
[0.1, 0.2, 0.3]


## The @cache Decorator

In [68]:
from functools import lru_cache
import time

@lru_cache(maxsize=1000)
def get_embedding(text):
    """Cache embedding results to avoid redundant API calls."""
    print(f"Computing embedding for: {text!r}")
    time.sleep(0.5)
    return len(text) / 1000

get_embedding("machine learning")
get_embedding("machine learning")
get_embedding("machine learning")

print(get_embedding.cache_info())
# CacheInfo(hits=1, misses=1, maxsize=1000, currsize=1)

Computing embedding for: 'machine learning'
CacheInfo(hits=2, misses=1, maxsize=1000, currsize=1)


In [69]:
from functools import wraps

def cache_embeddings(func):
    cache = {}

    @wraps(func)
    def wrapper(text, **kwargs):
        cache_key = (text, tuple(sorted(kwargs.items())))
        if cache_key not in cache:
            cache[cache_key] = func(text, **kwargs)
        return cache[cache_key]
    wrapper.cache = cache
    wrapper.cache_clear = lambda: cache.clear()
    return wrapper

@cache_embeddings
def embed(text, model="text-embedding-3-small"):
    print(f"Calling embedding model: {text[:30]}...")
    return [0.1, 0.2, 0.3]

embed("hello", model="text-embedding-3-small")
embed("hello", model="text-embedding-3-small")
embed("hello", model="text-embedding-3-large")
print(f"Cache size: {len(embed.cache)}")  # 2

Calling embedding model: hello...
Calling embedding model: hello...
Cache size: 2


## Stacking Decorators
timing wraps the function first, then retry wraps the timed function.

In [70]:
@retry(max_retries=3, delay=1.0, backoff=2.0)
@timing
def call_llm(prompt):
    pass

## Generators and yield

In [71]:
def count_up_to(n):
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(3)
print(next(gen))  # 1
print(next(gen))  # 2
print(next(gen))  # 3
# next(gen) would raise StopIteration


1
2
3


## Generator Expressions

In [75]:
# List comprehension: builds the full list now
squares_list = [x ** 2 for x in range(1_000_000)]

# Generator expression: computes values as they are requested
squares_gen = (x ** 2 for x in range(1_000_000))

for s in squares_gen:
    print(s)
    if s > 100:
        break

0
1
4
9
16
25
36
49
64
81
100
121


# Streaming LLM Responses

In [76]:
def stream_llm_response(prompt):
    chunks = [
        "RAG ",
        "combines ",
        "retrieval ",
        "with ",
        "generation.",
    ]

    for chunk in chunks:
        yield chunk

full_response = ""
for chunk in stream_llm_response("Explain RAG"):
    print(chunk, end="", flush=True)
    full_response += chunk

print()
print(f"Full response: {full_response}")

RAG combines retrieval with generation.
Full response: RAG combines retrieval with generation.


In a real application, prefer the provider SDK's streaming helpers when they exist.

In [ ]:
from openai import OpenAI

client = OpenAI()

def stream_response(prompt):
    """Yield text chunks from a streaming LLM call."""
    stream = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt,
        stream=True,
    )
    for event in stream:
        if event.type == "response.output_text.delta":
            yield event.delta

for token in stream_response("What is attention in transformers?"):
    print(token, end="", flush=True)

# Batching with Generators

In [82]:
def batch_items(iterable, batch_size):
    """Yield batches from any iterable."""
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

texts = [f"Document {i}" for i in range(10_000)]

for i, batch in enumerate(batch_items(texts, batch_size=500)):
    print(f"Batch {i}: {len(batch)} texts")

Batch 0: 500 texts
Batch 1: 500 texts
Batch 2: 500 texts
Batch 3: 500 texts
Batch 4: 500 texts
Batch 5: 500 texts
Batch 6: 500 texts
Batch 7: 500 texts
Batch 8: 500 texts
Batch 9: 500 texts
Batch 10: 500 texts
Batch 11: 500 texts
Batch 12: 500 texts
Batch 13: 500 texts
Batch 14: 500 texts
Batch 15: 500 texts
Batch 16: 500 texts
Batch 17: 500 texts
Batch 18: 500 texts
Batch 19: 500 texts


# itertools Essentials

## chain: Flatten Multiple Iterables

In [83]:
from itertools import chain

train_docs = ["doc1", "doc2", "doc3"]
val_docs = ["doc4", "doc5"]
test_docs = ["doc6"]

for doc in chain(train_docs, val_docs, test_docs):
    print(f"Processing: {doc}")

Processing: doc1
Processing: doc2
Processing: doc3
Processing: doc4
Processing: doc5
Processing: doc6


## islice: Take a Slice of Any Iterator

In [84]:
from itertools import islice

def infinite_prompts():
    i = 0
    while True:
        yield f"Prompt #{i}: Explain concept {i}"
        i += 1

for prompt in islice(infinite_prompts(), 5):
    print(prompt)

Prompt #0: Explain concept 0
Prompt #1: Explain concept 1
Prompt #2: Explain concept 2
Prompt #3: Explain concept 3
Prompt #4: Explain concept 4


## batched: Native Batching (Python 3.12+)

In [ ]:
from itertools import batched  # Python 3.12+

texts = ["text1", "text2", "text3", "text4", "text5"]

for batch in batched(texts, 2):
    print(batch)
# ('text1', 'text2')
# ('text3', 'text4')
# ('text5',)

# Lambda Functions

In [ ]:
results = [
    {"model": "large-model", "accuracy": 0.92, "cost": 0.03},
    {"model": "balanced-model", "accuracy": 0.95, "cost": 0.04},
    {"model": "small-local-model", "accuracy": 0.88, "cost": 0.001},
]

by_efficiency = sorted(results, key=lambda r: r["accuracy"] / r["cost"], reverse=True)
print(by_efficiency[0]["model"])  # small-local-model

accurate = list(filter(lambda r: r["accuracy"] > 0.90, results))
print([r["model"] for r in accurate])

scores = [0.7, 0.85, 0.92, 0.5]
labels = list(map(lambda s: "pass" if s >= 0.8 else "fail", scores))
print(labels)  # ['fail', 'pass', 'pass', 'fail']

small-local-model
['large-model', 'balanced-model']


# functools.partial

functools.partial creates a new function with some arguments already filled in. It is useful when you have a general function and want a few project-specific versions of it.

In [94]:
from functools import partial

def call_model(prompt, model, temperature=0.7, max_tokens=1000):
    return f"[{model} temp={temperature}] {prompt[:50]}..."

call_precise = partial(call_model, model="large-model", temperature=0.0)
call_fast = partial(call_model, model="small-model", temperature=0.3)
call_creative = partial(call_model, model="large-model", temperature=1.0)

print(call_precise("Explain binary search"))
# [large-model temp=0.0] Explain binary search...

print(call_creative("Write a poem about databases"))
# [large-model temp=1.0] Write a poem about databases...

[large-model temp=0.0] Explain binary search...
[large-model temp=1.0] Write a poem about databases...


In [95]:
from functools import partial

def embed_text(text, model="text-embedding-3-small", dimensions=256):
    return f"Embedded with {model} ({dimensions}d): {text[:30]}"

embed = partial(embed_text, model="text-embedding-3-large", dimensions=1024)

for doc in ["First document", "Second document", "Third document"]:
    print(embed(doc))

Embedded with text-embedding-3-large (1024d): First document
Embedded with text-embedding-3-large (1024d): Second document
Embedded with text-embedding-3-large (1024d): Third document
